<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Traders Autônomos</h2>
            <span style="color:#ff7800;">Uma simulação de negociação de ações para ilustrar agentes autônomos impulsionados por ferramentas e recursos de servidores MCP.
            </span>
        </td>
    </tr>
</table>

### Semana 6 Dia 4

E agora — apresentando o projeto Capstone:


# Traders Autônomos

Uma simulação de negociação de ações, com 4 Traders e um Pesquisador, impulsionada por uma variedade de servidores MCP com ferramentas e recursos:

1. Nosso servidor MCP de Contas feito em casa (escrito pela nossa equipe de engenharia!)
2. Fetch (obter página da web via um navegador local headless)
3. Memory
4. Brave Search
5. Dados financeiros

E um recurso para ler informações sobre a conta do trader e sua estratégia de investimento.

O objetivo do laboratório de hoje é criar um novo módulo Python, `traders.py`, que gerenciará um único trader em nosso piso de negociação.

Vamos experimentar e explorar no laboratório e, quando estivermos prontos, migraremos para um módulo Python.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Mais uma vez —</h2>
            <span style="color:#ff7800;">Por favor, não use isso para decisões reais de negociação!!
            </span>
        </td>
    </tr>
</table>

In [8]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

### Vamos começar reunindo os parâmetros MCP para nosso trader


In [9]:
polygon_api_key = os.getenv("POLYGON_API_KEY")
polygon_plan = os.getenv("POLYGON_PLAN")

is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

print(is_paid_polygon)
print(is_realtime_polygon)

False
False


In [10]:
if is_paid_polygon or is_realtime_polygon:
    market_mcp = {"command": "uvx","args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@master", "mcp_polygon"], "env": {"POLYGON_API_KEY": polygon_api_key}}
else:
    market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

### E agora para nosso pesquisador


In [11]:
brave_env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}

researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": brave_env}
]

### Agora crie o MCPServerStdio para cada um


In [12]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

### Agora vamos criar um Agente Pesquisador para fazer pesquisa de mercado

E transformá-lo em uma ferramenta — lembre como isso funciona no OpenAI Agents SDK e a diferença em relação aos handoffs?


In [13]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""Você é um pesquisador financeiro. Você pode pesquisar na web por notícias financeiras interessantes,
procurar possíveis oportunidades de negociação e ajudar com pesquisas.
Com base no pedido, realize as pesquisas necessárias e responda com suas conclusões.
Reserve tempo para fazer várias buscas para obter uma visão abrangente e depois resuma suas conclusões.
Se não houver um pedido específico, responda com oportunidades de investimento com base na pesquisa das últimas notícias.
A data e hora atuais são {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model="gpt-5-mini",
        mcp_servers=mcp_servers,
    )
    return researcher

In [14]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="Esta ferramenta pesquisa online por notícias e oportunidades, \
                seja a partir do seu pedido específico para analisar uma determinada ação, \
                ou de forma geral por notícias e oportunidades financeiras relevantes. \
                Descreva que tipo de pesquisa você está buscando."
        )

In [15]:
research_question = "Quais são as últimas notícias sobre a Amazon?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Aqui estão as principais notícias recentes sobre a Amazon (resumo das últimas semanas — 2025-10-03 a 2025-10-12):

1) Prime Big Deal Days (outubro 7–8, 2025) — evento e resultados
- A Amazon promoveu o “Prime Big Deal Days” em 7–8 de outubro como prévia da temporada de fim de ano. A empresa afirmou que foi o maior evento do tipo (novos recordes de vendas e poupanças para clientes) e destacou várias promoções. (Fonte: comunicado da Amazon / AboutAmazon)
- Cobertura crítica/analítica aponta que, embora as vendas tenham sido altas, o comportamento do consumidor mostrou sinais mistos — por exemplo, queda no ticket médio vs. Prime Day de julho, sugerindo consumo mais “essencial/prático”. (Fonte: Forbes)

2) AWS e generative AI — lançamentos e métricas
- A AWS tem lançado várias atualizações para impulsionar aplicações de IA generativa: Amazon Bedrock ganhou recursos/novas ofertas e foi promovido como solução para construir agentes e modelos, além de integrar modelos de terceiros (incl. OpenAI/Anthropic/etc.). (Fonte: AWS / AboutAmazon)
- Relatos da própria AWS sobre o pico de uso na temporada de promoções mostram métricas de escala muito altas (ex.: picos massivos de invocações Lambda e tráfego em serviços gerenciados) — reforçando papel crítico da AWS em grandes eventos. (Fonte: AWS blog)

3) Resultados financeiros / outlook
- Relatórios trimestrais de 2025 continuam mostrando crescimento de receita (ex.: vendas líquidas do 2º trimestre +13% ano a ano em algumas comunicações da empresa). Ao mesmo tempo, investidores já vinham reagindo de forma sensível ao crescimento da nuvem (AWS) — houve momentos em que resultados/guia abaixo das expectativas fizeram a ação cair. (Fontes: comunicados de resultados da Amazon / Reuters)

4) Movimentações de executivos
- Notícia recente (Reuters, início de outubro) aponta saída de um executivo sênior da área de devices (VP de Devices), membro do “S-team” — movimento chamou atenção por ocorrer logo após lançamentos de produtos. (Fonte: Reuters)

5) Riscos regulatórios (contexto)
- A União Europeia e outras autoridades têm mantido atenção em práticas da Amazon (uso de dados de terceiros, tratamento de vendedores, Buy Box, etc.). Há históricos de investigação/compromissos; esse risco regulatório permanece um fator a observar, embora não haja nesta janela uma nova decisão final anunciada. (Fontes: comunicados da Comissão Europeia / cobertura especializada)

Implicações rápidas
- Varejo: o evento de outubro dá impulso para o 4º trimestre, mas sinais de ticket médio menor podem indicar consumidores mais cautelosos.
- AWS/IA: a Amazon está apostando pesado em ferramentas e produtos de IA generativa — isto é potencialmente um motor de receita e diferencial competitivo, mas também sujeita a competição intensa e escrutínio por performance/precificação.
- Ações/regulação: resultados e guias da AWS continuam sendo gatilhos para volatilidade das ações; riscos regulatórios na UE e nos EUA seguem como variáveis a monitorar.

Quer que eu:
- Busque artigos completos (links/leituras) das fontes citadas (Reuters, Forbes, AboutAmazon, AWS blog)?
- Faça uma análise do impacto dessas notícias no papel AMZN (curto/prazo/ligado a preço)?
- Foque só em AWS/IA, varejo ou regulação com mais profundidade?

### Observe o trace

https://platform.openai.com/traces

In [16]:
ed_initial_strategy = "Você é um day trader que compra e vende ações de forma agressiva com base em notícias e condições de mercado."
Account.get("Ed").reset(ed_initial_strategy)

display(Markdown(await read_accounts_resource("Ed")))
display(Markdown(await read_strategy_resource("Ed")))

{"name": "ed", "balance": 10000.0, "strategy": "Voc\u00ea \u00e9 um day trader que compra e vende a\u00e7\u00f5es de forma agressiva com base em not\u00edcias e condi\u00e7\u00f5es de mercado.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-10-12 11:47:48", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

Você é um day trader que compra e vende ações de forma agressiva com base em notícias e condições de mercado.

### E agora - para criar nosso Agente Trader


In [17]:
agent_name = "Ed"

# Usando servidores MCP para ler recursos
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)

instructions = f"""
Você é um trader que gerencia uma carteira de ações. Seu nome é {agent_name} e sua conta está em seu nome, {agent_name}.
Você tem acesso a ferramentas que permitem pesquisar na internet notícias sobre empresas, verificar preços de ações e comprar e vender ações.
Sua estratégia de investimento para sua carteira é:
{strategy}
Suas posições atuais e saldo são:
{account_details}
Você tem ferramentas para realizar buscas na web por notícias e informações relevantes.
Você tem ferramentas para verificar preços de ações.
Você tem ferramentas para comprar e vender ações.
Você tem ferramentas para salvar memórias sobre empresas, pesquisas e seu raciocínio até o momento.
Use essas ferramentas para gerenciar sua carteira. Realize negociações como achar adequado; não espere instruções nem peça confirmação.
"""

prompt = """
Use suas ferramentas para tomar decisões sobre sua carteira.
Investigue as notícias e o mercado, tome sua decisão, execute as negociações e responda com um resumo das suas ações.
"""

In [18]:
print(instructions)


Você é um trader que gerencia uma carteira de ações. Seu nome é Ed e sua conta está em seu nome, Ed.
Você tem acesso a ferramentas que permitem pesquisar na internet notícias sobre empresas, verificar preços de ações e comprar e vender ações.
Sua estratégia de investimento para sua carteira é:
Você é um day trader que compra e vende ações de forma agressiva com base em notícias e condições de mercado.
Suas posições atuais e saldo são:
{"name": "ed", "balance": 10000.0, "strategy": "Voc\u00ea \u00e9 um day trader que compra e vende a\u00e7\u00f5es de forma agressiva com base em not\u00edcias e condi\u00e7\u00f5es de mercado.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-10-12 11:47:48", 10000.0], ["2025-10-12 11:47:54", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}
Você tem ferramentas para realizar buscas na web por notícias e informações relevantes.
Você tem ferramentas para verificar preços de ações.
Você tem ferramentas para c

### E para executar nosso Trader


In [19]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool],
    mcp_servers=trader_mcp_servers,
    model="gpt-5-mini",
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

Resumo das ações que executei agora (conta: Ed)

Observação rápida sobre a pesquisa: tentei rodar uma varredura de notícias em tempo real (premarket/after‑hours) com a ferramenta de pesquisa, mas houve erro na execução das consultas. Em seguida verifiquei preços e executei trades com base em fluxo de mercado e volatilidade observada.

Preços verificados (consulta rápida):
- NVDA: ~183.16 (preço de referência)
- GME: ~23.30 (preço de referência)
- Outras cotações consultadas: TSLA, AAPL, AMD, PLTR, MSFT (para contexto)

Negociações executadas (agressivo / day‑trading):
1) Comprado NVDA — 25 ações
   - Preço de execução: 183.52632
   - Racional: NVDA é líquida e tende a ter movimentos intradiários fortes; posicionei ~45% do capital (em linha com estratégia agressiva) para capturar rali de curto prazo / scalps. Vou usar stops e metas de lucro intradiárias e ajustar conforme fluxo de notícias/momentum.

2) Comprado GME — 231 ações
   - Preço de execução: 23.3466
   - Racional: GME apresenta alta volatilidade e volume intradiário, boa para scalps rápidos. Aloquei o saldo disponível restante para exposição de alta oscilação, com gestão ativa de risco.

Estado da conta após as operações:
- Saldo em caixa: 18.7774 (aprox.)
- Posições abertas:
  - NVDA: 25 ações @183.52632
  - GME: 231 ações @23.3466
- Valor total de portfólio (sistema): ~9,980.08 (ref. momento da execução)
- Histórico: compras registradas com rationale (conforme acima)

Plano de gestão intradiária (próximos passos — execução ativa):
- Monitorar notícias e fluxo de ordens em tempo real (se surgir notícia relevante ajusto ou fecho posições).
- Stops iniciais (intradiários, ajustáveis com o mercado):
  - NVDA: stop inicial ~-2.5% do preço de execução (ajustar se houver catalisadores)
  - GME: stop inicial ~-3% (volatilidade maior, dar espaço mas com disciplina)
- Metas de lucro / escalonamento:
  - Realizar lucros parciais em +3% a +5% (dependendo do ritmo) e mover stops para breakeven.
  - Buscar múltiplos scalps; fechar posições antes do fechamento do pregão (estrita regra day‑trade).
- Se volumes e notícias indicarem continuação forte, posso aumentar exposições parcialmente (seguindo gerenciamento de risco). Caso contrário, vou reduzir exposição rapidamente.

Observações finais:
- Tentei pesquisa de notícias em tempo real, mas ocorreu erro na ferramenta; manterei tentativa de nova varredura e monitorarei simultaneamente feeds de notícias e nível de livros de ordens.
- Se preferir, posso enviar atualizações periódicas (ex.: a cada 30–60 minutos) sobre P&L, alterações de stops e eventuais fechamentos.

Quer que eu rode novamente a pesquisa de notícias agora e integre achados com ajustes de posição?

### Depois vá olhar o trace

http://platform.openai.com/traces


In [20]:
# E vamos ver os resultados da negociação

await read_accounts_resource(agent_name)

'{"name": "ed", "balance": 18.777399999999034, "strategy": "Voc\\u00ea \\u00e9 um day trader que compra e vende a\\u00e7\\u00f5es de forma agressiva com base em not\\u00edcias e condi\\u00e7\\u00f5es de mercado.", "holdings": {"NVDA": 25, "GME": 231}, "transactions": [{"symbol": "NVDA", "quantity": 25, "price": 183.52632, "timestamp": "2025-10-12 11:52:33", "rationale": "NVDA com pre\\u00e7o de 183.16 \\u2014 ativo l\\u00edquido com forte volatilidade intradi\\u00e1ria. Posicionamento curto para day trade buscando aproveitar movimentos de not\\u00edcia/momentum; exposi\\u00e7\\u00e3ode ~45% do capital para capturar potencial rali de curto prazo. Ajustarei stop intradi\\u00e1rio e realizarei lucro r\\u00e1pido conforme fluxo de mercado."}, {"symbol": "GME", "quantity": 231, "price": 23.346600000000002, "timestamp": "2025-10-12 11:52:48", "rationale": "GME com pre\\u00e7o de ~23.3 \\u2014 alto volume e volatilidade intradi\\u00e1ria. Alocar o restante do capital em um ticker de alta osci

### Agora é hora de revisar o módulo Python criado a partir disto:

`mcp_params.py` é onde os servidores MCP são especificados. Você vai perceber que trouxe alguns velhos conhecidos: memória e notificações push!

`templates.py` é onde as instruções e mensagens são configuradas (isto é, os prompts de Sistema e de Usuário)

`traders.py` reúne tudo.

Você vai notar que fiz algo um pouco sofisticado com um código como este:

```
async with AsyncExitStack() as stack:
    mcp_servers = [await stack.enter_async_context(MCPServerStdio(params)) for params in mcp_server_params]
```

Essa é apenas uma maneira organizada de combinar nossas declarações "with" (conhecidas como gerenciadores de contexto) para que não precisemos fazer algo feio como isto:

```
async with MCPServerStdio(params=params1) as mcp_server1:
    async with MCPServerStdio(params=params2) as mcp_server2:
        async with MCPServerStdio(params=params3) as mcp_server3:
            mcp_servers = [mcp_server1, mcp_server2, mcp_server3]
```

Mas é equivalente.


In [21]:
from traders import Trader


In [22]:
trader = Trader("Ed")

In [23]:
await trader.run()

In [24]:
await read_accounts_resource("Ed")

'{"name": "ed", "balance": 716.379399999999, "strategy": "Voc\\u00ea \\u00e9 um day trader que compra e vende a\\u00e7\\u00f5es de forma agressiva com base em not\\u00edcias e condi\\u00e7\\u00f5es de mercado.", "holdings": {"NVDA": 25, "GME": 201}, "transactions": [{"symbol": "NVDA", "quantity": 25, "price": 183.52632, "timestamp": "2025-10-12 11:52:33", "rationale": "NVDA com pre\\u00e7o de 183.16 \\u2014 ativo l\\u00edquido com forte volatilidade intradi\\u00e1ria. Posicionamento curto para day trade buscando aproveitar movimentos de not\\u00edcia/momentum; exposi\\u00e7\\u00e3ode ~45% do capital para capturar potencial rali de curto prazo. Ajustarei stop intradi\\u00e1rio e realizarei lucro r\\u00e1pido conforme fluxo de mercado."}, {"symbol": "GME", "quantity": 231, "price": 23.346600000000002, "timestamp": "2025-10-12 11:52:48", "rationale": "GME com pre\\u00e7o de ~23.3 \\u2014 alto volume e volatilidade intradi\\u00e1ria. Alocar o restante do capital em um ticker de alta oscila

### Agora observe o trace

https://platform.openai.com/traces

### Quantas ferramentas usamos no total?

In [25]:
from mcp_params import trader_mcp_server_params, researcher_mcp_server_params

all_params = trader_mcp_server_params + researcher_mcp_server_params("ed")

count = 0
for each_params in all_params:
    async with MCPServerStdio(params=each_params, client_session_timeout_seconds=60) as server:
        mcp_tools = await server.list_tools()
        count += len(mcp_tools)
print(f"Temos {len(all_params)} servidores MCP e {count} ferramentas")


Temos 6 servidores MCP e 16 ferramentas
